In [ ]:
import torch
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt
import math
import torch.nn as nn
import torch.nn.functional as F

# Inference-Time Sampling for Rescaling Distribution Temperature

We design an inference-time sampler for sharpening and flattening target distributions. Our sampler works with standard pre-trained DDPM models.

## Define parameters

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_diffusion_steps = 100

datasets = {
    "single":   {"dataset_shape": (50_000,1),  "means": [0.0],            "stds": 1.0},
    "barrier":  {"dataset_shape": (50_000,1),  "means": [-3.0, 3.0],      "stds": 0.5},
    "composed": {"dataset_shape": (100_000,1), "means": [-3.0, 0.0, 3.0], "stds": [0.5, 1.0, 0.5]}
}

ckpt_dir = "model_checkpoints"

## Generating and Visualizing Samples

We have two functions here: ```generate_gaussian_mixture``` samples according to the dataset configuration provided by the datasets dictionary defined above, and ```compute_mixture_pdf``` generates the analytical probability density function, similarly from the dataset name above. All Gaussians have equal weights in the mixture.

In [ ]:
@torch.no_grad()
def generate_gaussian_mixture(dataset_name, device='cpu'):
	"""Generates mixture of gaussians according to inputted means and standard deviations"""

	dataset_config = datasets[dataset_name]
	dataset_shape = dataset_config["dataset_shape"]
	n_samples = dataset_shape[0]

	means = torch.as_tensor(dataset_config['means'], dtype=torch.float32)
	stds = dataset_config['stds']
	
	n_gaussians = len(means)
	
	if isinstance(stds, (int, float)):
		stds = torch.full((n_gaussians,), float(stds))
	else:
		stds = torch.as_tensor(stds, dtype=torch.float32)
		assert len(stds) == n_gaussians, f"stds length {len(stds)} != n_gaussians {n_gaussians}"
		
	component_ids = np.random.choice(n_gaussians, size=n_samples)
	samples = torch.zeros(n_samples, 1, device=device)
	
	for i in range(n_gaussians):
		mask = component_ids == i
		samples[mask] = torch.normal(
			mean=float(means[i]),
			std=float(stds[i]),
			size=(mask.sum(), 1)
		).to(device)
	
	return samples


@torch.no_grad()
def compute_mixture_pdf(dataset_name, x_axis, k=1.0):
	"""Computes analytical pdf of training dataset from dataset config file, used for plotting"""

	dataset_config = datasets[dataset_name]
	means = np.array(dataset_config['means'])
	stds = dataset_config['stds']
		
	if isinstance(stds, (int, float)):
		stds = np.full(len(means), stds)
	else:
		stds = np.array(stds)
		
	stds = stds / np.sqrt(k)
	
	n_gaussians = len(means)
	pdf = np.zeros_like(x_axis)
	
	for mu, sigma in zip(means, stds):
		pdf += norm.pdf(x_axis, loc=mu, scale=sigma)
	
	pdf /= n_gaussians  
	
	return pdf 

We can visualize our samples and pdf for a given template distribution.

In [ ]:
def plot_samples(datast_name, x_limit=10, n_bins=200):

	x_axis = np.linspace(-x_limit, x_limit, n_bins)
	bins = np.linspace(-x_limit, x_limit, n_bins)

	samples = generate_gaussian_mixture("single")
	pdf = generate_gaussian_mixture("single", x_axis)

	plt.hist(samples, bins=bins, density=True)
	plt.plot(pdf)

# Noise Schedule + TSR Implementation

We follow the DDPM noise schedule.

In [ ]:
@torch.no_grad()
def cosine_beta_schedule(timesteps, s=0.008):
	"""Cosine noise schedule, taken from reduce reuse recycle code"""
	steps = timesteps + 1
	t = torch.linspace(0, timesteps, steps, dtype=torch.float32) / timesteps
	alphas_cumprod = torch.cos((t + s) / (1 + s) * math.pi * 0.5) ** 2
	alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
	betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
	return torch.clip(betas, 0, 0.999)

betas = cosine_beta_schedule(n_diffusion_steps).to(device)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)  # alphā_t

ts_desc = torch.arange(n_diffusion_steps - 1, -1, -1, device=device)

We follow the implementation of TSR

In [ ]:
@torch.no_grad()
def compute_tsr_schedule(k, sigma, t):
	"""Computes temporal score rescaling coefficient. Outpit will be shape (N_DIFFUSION_STEPS, n_langevin_steps)"""

	a_bar = alpha_bars[t]
	sigma_t = torch.sqrt(1.0 - a_bar)
	alpha_t = torch.sqrt(a_bar)

	eta_t = (alpha_t**2) / (sigma_t**2)
	num = eta_t * (sigma ** 2) + 1
	den = (eta_t * (sigma ** 2)) / k + 1
	tsr = num / den

	return tsr

# We follow the DDPM Logic

In [ ]:
@torch.no_grad()
def compute_score(model, x, t, k, sigma):
	"""Computes score = - epsilon * temp / √(1 - α_bar)"""
	
	x_shape = x.shape
	ones = torch.ones((x_shape[0], 1), device=device)
	eps_hat = model(x, t * ones)   
	a_bar = alpha_bars[t]
	temp_t = compute_tsr_schedule(k, sigma, t)
	score_hat= - eps_hat * temp_t / torch.sqrt(1.0 - a_bar)
	return score_hat

In [ ]:
@torch.no_grad()
def ddpm_tsr(model, dataset_shape, k=1.0, sigma=1.0):
	"""Sampling algorithm for DDPM, ULA, and MALA"""

	x = torch.randn(dataset_shape, device=device)
		
	for t in ts_desc: 

		alpha_t = alphas[t]
		beta_t = betas[t]
		sqrt_alpha_t = torch.sqrt(alpha_t)
		sqrt_beta_t = torch.sqrt(beta_t)
		noise = torch.randn(dataset_shape, device=device)

		score_hat = compute_score(model, x, t, k, sigma)
		x = (x + beta_t * score_hat) / sqrt_alpha_t + sqrt_beta_t * noise

	return x

# Model

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
	def __init__(self, dim):
		super().__init__()
		self.dim = dim

	def forward(self, t):
		if t.dim() == 2:
			t = t.squeeze(-1)

		half = self.dim // 2
		freqs = torch.exp(
			-math.log(10000) * torch.arange(half, device=t.device) / half
		)
		args = t[:, None] * freqs[None, :]
		emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

		if self.dim % 2 == 1:
			emb = F.pad(emb, (0, 1))

		return emb


class MLP(nn.Module):
    def __init__(
        self,
        x_dim=1,
        hidden_dim=512,   # 128 -> 512
        time_dim=64,      # 32 -> 64
        n_layers=8,       # 4 -> 8
    ):
        super().__init__()

        self.time_embed = SinusoidalTimeEmbedding(time_dim)
        self.input = nn.Linear(x_dim + time_dim, hidden_dim)
        self.layers = nn.ModuleList(
            [nn.Linear(hidden_dim, hidden_dim) for _ in range(n_layers)]
        )
        self.output = nn.Linear(hidden_dim, x_dim)

    def forward(self, x, t):
        t_emb = self.time_embed(t)
        h = torch.cat([x, t_emb], dim=-1)
        h = F.silu(self.input(h))
        for layer in self.layers:
            h = h + F.silu(layer(h))
        return self.output(h)


In [ ]:
def load_model(path):
	"""Load trained model from checkpoint"""
	model = MLP().to(device)
	model.load_state_dict(torch.load(path, map_location=device))
	model.eval()
	return model

# Sampling

In [ ]:
for dataset_name in datasets.keys():

	model = load_model(f"{ckpt_dir}/{dataset_name}_1.0.pt") # dataset name would always be first if it is a param
	dataset_config = datasets[dataset_name]
	dataset_shape = dataset_config["dataset_shape"]

	samples = ddpm_tsr(model, dataset_shape, k=1.0, sigma=1.0)